

# a Advanced Pipelines Regression
### OPIM 5512 — Applied Data Science · Module2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module2/scripts__1a_AdvancedPipelines_Regression.ipynb)

*Run me top to bottom — **Runtime → Run all**. Data loads from a stable link, so there's nothing to upload.*

# Advanced Pipelines with Grid Search (regression)
--------------
**OPIM 5512 - Data Science Using Python** - University of Connecticut

Rather than using pipelines to evaluate the models with defaults and THEN performing your grid search - why don't we try to do everything at once?!

# Import Modules

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
#from sklearn.externals import joblib
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [2]:
# Load dataset
# we will use Gdown to load our Boston Housing dataset
# https://drive.google.com/file/d/1a0aNGSFWB-pf5ut1NsjE5ECIsbHHoAwI/view?usp=sharing
!gdown 1a0aNGSFWB-pf5ut1NsjE5ECIsbHHoAwI

# look left! it downloaded a local copy of 'BostonHousing.csv'

Downloading...
From: https://drive.google.com/uc?id=1a0aNGSFWB-pf5ut1NsjE5ECIsbHHoAwI
To: /content/BostonHousing.csv
100% 35.2k/35.2k [00:00<00:00, 63.0MB/s]


In [3]:
from sklearn.datasets import fetch_california_housing
df = fetch_california_housing(as_frame=True).frame
df = df.sample(n=3000, random_state=42).reset_index(drop=True)  # subsample for snappy grid search
df.head()

## Read Data

In [4]:
# Split-out validation df
X = df.drop('MedHouseVal', axis=1) #covariates - just drop the target!
y = df['MedHouseVal'] #target variable
validation_size = 0.20
seed = 123 # so you will split the same way and evaluate the SAME dataset

# split!
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                                test_size=validation_size,
                                                                random_state=seed)

## Build Pipeline

In [5]:
# Construct some pipelines
pipe_dt = Pipeline([('scl', StandardScaler()),
			('clf', DecisionTreeRegressor(random_state=42))])

pipe_dt_pca = Pipeline([('scl', StandardScaler()),
			('pca', PCA(0.95)),
			('clf', DecisionTreeRegressor(random_state=42))])

pipe_rf = Pipeline([('scl', StandardScaler()),
			('clf', RandomForestRegressor(random_state=42))])

pipe_rf_pca = Pipeline([('scl', StandardScaler()),
			('pca', PCA(0.95)),
			('clf', RandomForestRegressor(random_state=42))])


## Define your Parameters for Grid Search

In [6]:
# Set grid search params
param_range = [5, 10, 15, 25]

# be careful! you need clf with two underscores clf__,
# because this is how we named the model above (look at all the clf)
# if you change clf above, make sure you update it down here too

grid_params_dt = [{
		'clf__min_samples_leaf': param_range,
		'clf__max_depth': param_range,
		'clf__min_samples_split': param_range[1:]}] #everything except the first one!

grid_params_rf = [{
		'clf__min_samples_leaf': param_range,
		'clf__max_depth': param_range,
		'clf__min_samples_split': param_range[1:]}] #everything except the first one!

## Define your Grid Search

In [7]:
# Construct grid searches

gs_dt = GridSearchCV(estimator=pipe_dt,
    param_grid=grid_params_dt,
    scoring='neg_median_absolute_error',
    cv=10)

gs_dt_pca = GridSearchCV(estimator=pipe_dt_pca,
    param_grid=grid_params_dt,
    scoring='neg_median_absolute_error',
    cv=10)

gs_rf = GridSearchCV(estimator=pipe_rf,
    param_grid=grid_params_rf,
    scoring='neg_median_absolute_error',
    cv=10)

gs_rf_pca = GridSearchCV(estimator=pipe_rf_pca,
    param_grid=grid_params_rf,
    scoring='neg_median_absolute_error',
    cv=10)


# List of pipelines for ease of iteration
grids = [gs_dt, gs_dt_pca, gs_rf, gs_rf_pca]

# Dictionary of pipelines and classifier types for ease of reference
grid_dict = {0: 'Decision Tree', 1: 'Decision Tree w/PCA',
  2: 'Random Forest', 3: 'Random Forest w/PCA'}


**On your own:** Can you calculate how many models we are going to fit? 256! Can you do the math and get the same answer?

## Run it! Find the best model

In [8]:
from sklearn.metrics import median_absolute_error

In [9]:

# Fit the grid search objects
print('Performing model optimizations...')
best_err = np.inf
best_clf = 0
best_gs = ''
for idx, gs in enumerate(grids):
	print('\nEstimator: %s' % grid_dict[idx])
	# Fit grid search
	gs.fit(X_train, y_train)
	# Best params
	print('Best params: %s' % gs.best_params_)
	# Best training data error
	print('Best training error: %.3f' % gs.best_score_)
	# Predict on test data with best params
	y_pred = gs.predict(X_test)
	# Test data error of model with best params
	print('Test set error score for best params: %.3f ' % median_absolute_error(y_test, y_pred))
	# Track best (lowest test error) model
	if median_absolute_error(y_test, y_pred) < best_err: #updated: April 8, 2021 (4 PM)
		best_err = median_absolute_error(y_test, y_pred)
		best_gs = gs
		best_clf = idx
print('\nModel with best test set error: %s' % grid_dict[best_clf])

Performing model optimizations...

Estimator: Decision Tree
Best params: {'clf__max_depth': 10, 'clf__min_samples_leaf': 5, 'clf__min_samples_split': 15}
Best training error: -1.931
Test set error score for best params: 2.277 

Estimator: Decision Tree w/PCA
Best params: {'clf__max_depth': 10, 'clf__min_samples_leaf': 5, 'clf__min_samples_split': 10}
Best training error: -2.287
Test set error score for best params: 1.770 

Estimator: Random Forest
Best params: {'clf__max_depth': 10, 'clf__min_samples_leaf': 5, 'clf__min_samples_split': 10}
Best training error: -1.786
Test set error score for best params: 1.438 

Estimator: Random Forest w/PCA
Best params: {'clf__max_depth': 15, 'clf__min_samples_leaf': 5, 'clf__min_samples_split': 15}
Best training error: -2.005
Test set error score for best params: 2.242 

Model with best test set error: Random Forest


Remember - in a classification problem, we want accuracy to be 0 and then improve towards 1.

* best_err = 0
* if accuracy(y_test, y_pred) > best_err:


In a regression problem, we want error to be infinity at the start, and then improve towards 0. Be careful!

* best_err = np.inf
* if accuracy(y_test, y_pred) > best_err:

# Now what? Refit the model and evaluate it!
So now you know the combination of hyperparameters that will yield an accurate model. You should:

* Re-run the model
* Store the train and test preds
* Make scatterplots and calculate error metrics
* Tell a story!

Note here that we have only fit two models with different scaling techniques - why not go try ALL of the other regression models we showed during spot check models? Maybe try calculating some [polynomial features/interaction terms](https://machinelearningmastery.com/polynomial-features-transforms-for-machine-learning/) in the pipeline as well? Or expanding the grid search to try different hyperparameters?



In [10]:
# left to students to do as an exercise